# Azure Auto ML for Image Data

# Notebook Setup

In [2]:
import os
# Here we set the working directory to the project root to ensure imports work correctly
from pathlib import Path, os
target = "dp100-learn"
p = Path.cwd()
print(f"Starting working directory: {p}")
while p.name != target and p.parent != p:
    p = p.parent
# Set the path to your project root manually if the above code does not work
p = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn"
# p = "C:/Users/dmika/DEV/Projects-local/dp100-learn"
os.chdir(p)
print("Changed working directory to:", p)
from utils.azureml_utils import *

# Get Azure ML Client based on your environment. Learn more in the tutorials/azureml-first-notebook.ipynb.
ml_client = get_azureml_client()

Starting working directory: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code
Changed working directory to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn
Added to sys.path: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn


Found the config file in: /config.json


Added to sys.path: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn


# Prepare Data

In [3]:
dataset_dir = Path(os.path.join(project_dir, "data/intel-image-classification"))
subset_dir = dataset_dir / "subset"
subset_dir.mkdir(exist_ok=True)
subset_dir = subset_dir / "data"
subset_dir.mkdir(exist_ok=True)

## Download and extract the data locally

In [ ]:
# Download and unzip dataset from Kaggle
import kaggle

# First you need to authenticate with Kaggle API by following instructions here: https://www.kaggle.com/docs/api.
kaggle.api.authenticate()
# Download the Intel Image Classification dataset and unzip it
kaggle.api.dataset_download_files('puneet6060/intel-image-classification', path=dataset_dir, unzip=True)

Dataset URL: https://www.kaggle.com/datasets/puneet6060/intel-image-classification


## Rearange the data into correct folder structure

In [4]:
# Flatten directory structure
import shutil

base = dataset_dir
for folder in ["seg_train", "seg_test", "seg_pred"]:
    inner = base / folder / folder
    if inner.exists():
        for item in inner.iterdir():
            shutil.move(str(item), str(base / folder))
        shutil.rmtree(inner)

## Create a subset for faster experimentation (optional)

In [ ]:
# Take a subset of the training data for quicker experiments
import random

for class_dir in (dataset_dir / "seg_train").iterdir():
    if class_dir.is_dir():
        files = list(class_dir.glob("*.jpg"))
        sample_files = random.sample(files, min(200, len(files)))
        target_dir = subset_dir / class_dir.name
        target_dir.mkdir(exist_ok=True)
        for f in sample_files:
            shutil.copy(f, target_dir / f.name)
print(f"Subset created at: {subset_dir}")

Subset created at: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn/data/intel-image-classification/subset/data


## Create a data asset

In [6]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

my_data = Data(
    path=str(subset_dir),
    datastore="dmdp100",
    type=AssetTypes.URI_FOLDER,
    description="Subset of Intel Image Classification dataset for AutoML image training",
    name="intel-image-subset-folder",
)
uploaded_data = ml_client.data.create_or_update(my_data)

Uploading data (18.29 MBs): 100%|██████████| 18294253/18294253 [00:11<00:00, 1644780.82it/s]




## Create annotation JSONL file 

### Relative Cloud path

In [13]:
import json

# img_data_asset = ml_client.data.get("intel-image-subset-folder", version="1")
# base_uri = img_data_asset.path
base_uri = "/data"

annotations_dir = subset_dir.parent
annotations_dir.mkdir(exist_ok=True)
jsonl_path = annotations_dir / "train_annotations.jsonl"
records = []

for class_dir in subset_dir.iterdir():
    if class_dir.is_dir():
        label = class_dir.name
        for img in class_dir.glob("*.jpg"):
            rel = img.relative_to(subset_dir)
            records.append({"image_url": f"{base_uri}/{rel.as_posix()}", "label": label})

with open(jsonl_path, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

print("✅ JSONL created:", jsonl_path)

✅ JSONL created: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn/data/intel-image-classification/subset/train_annotations.jsonl


### Full Cloud path

In [ ]:
import json

img_data_asset = ml_client.data.get("intel-image-subset-folder", version="1")
base_uri = img_data_asset.path

annotations_dir = subset_dir.parent / "annotations"
annotations_dir.mkdir(exist_ok=True)
jsonl_path = annotations_dir / "train_annotations.jsonl"
records = []

for class_dir in subset_dir.iterdir():
    if class_dir.is_dir():
        label = class_dir.name
        for img in class_dir.glob("*.jpg"):
            rel = img.relative_to(subset_dir)
            records.append({"image_url": f"{base_uri}{rel.as_posix()}", "label": label})

with open(jsonl_path, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

print("✅ JSONL created:", jsonl_path)

✅ JSONL created: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn/data/intel-image-classification/subset/annotations/train_annotations.jsonl


### Local

In [ ]:

# import json

# subset_dir = dataset_dir / "subset"
# jsonl_path = subset_dir / "train_annotations.jsonl"

# records = []

# for class_dir in subset_dir.iterdir():
#     if class_dir.is_dir():
#         label = class_dir.name
#         for img_path in class_dir.glob("*.jpg"):
#             record = {
#                 "image_url": str(img_path.resolve()),  # full path
#                 "label": label
#             }
#             records.append(record)

# # Write to JSONL
# with open(jsonl_path, "w", encoding="utf-8") as f:
#     for r in records:
#         f.write(json.dumps(r) + "\n")

# print(f"✅ JSONL created at: {jsonl_path}")
# print(f"Total records: {len(records)}")

✅ JSONL created at: C:\Users\dmika\DEV\Projects-local\dp100-learn\data\intel-image-classification\subset\train_annotations.jsonl
Total records: 1200


## Create MLTable with the annotations

In [14]:
%%writefile data/intel-image-classification/subset/MLTable

paths:
  - file: ./train_annotations.jsonl
transformations:
  - read_json_lines:
        encoding: utf8
        invalid_lines: error
        include_path_column: false
  - convert_column_types:
      - columns: image_url
        column_type: stream_info

Writing data/intel-image-classification/subset/MLTable


In [15]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

mltable_asset = Data(
    name="intel-image-subset-mltable",
    path=str(annotations_dir),
    type=AssetTypes.MLTABLE,
    datastore="dmdp100",
    description="Intel Image Classification subset prepared for AutoML"
)
registered_mltable = ml_client.data.create_or_update(mltable_asset)
print("✅ Registered MLTable asset:", registered_mltable.id)


Uploading subset (18.37 MBs): 100%|██████████| 18365832/18365832 [00:11<00:00, 1617206.93it/s]




✅ Registered MLTable asset: /subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/data/intel-image-subset-mltable/versions/2


# Running Auto ML for Image Classification

In [17]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import Input

# creates a dataset based on the files in the local data folder
my_training_data_input = Input(type=AssetTypes.MLTABLE, path="azureml:intel-image-subset-mltable:2")

# Training MLTable defined locally, with local data to be uploaded
# my_training_data_input = Input(type=AssetTypes.MLTABLE, path=training_mltable_path)
# Validation MLTable defined locally, with local data to be uploaded
# my_validation_data_input = Input(type=AssetTypes.MLTABLE, path=validation_mltable_path)
# WITH REMOTE PATH: If available already in the cloud/workspace-blob-store
# my_training_data_input = Input(type=AssetTypes.MLTABLE, path="azureml://datastores/workspaceblobstore/paths/vision-classification/train")
# my_validation_data_input = Input(type=AssetTypes.MLTABLE, path="azureml://datastores/workspaceblobstore/paths/vision-classification/valid")

In [18]:
from azure.ai.ml import automl

image_classification_job = automl.image_classification(
    compute="dmdp100-gpu-cluster",
    experiment_name="dmdp100-automl-img-classification",
    display_name="intel-imgs-subset-classification-automl",
    training_data=my_training_data_input,
    target_column_name="label"
)

In [19]:
# Set limits
image_classification_job.set_limits(
    timeout_minutes=120,
    max_trials=10,
    max_concurrent_trials=3,
)

In [20]:
# Submit the AutoML job
returned_job = ml_client.jobs.create_or_update(
    image_classification_job
)  